In [1]:
#Chronological Split 
import pandas as pd
import numpy as np

model_df = pd.read_csv("../data/processed/model_table.csv",
                       parse_dates=["week"], dtype={"StockCode": str})

train = model_df[model_df["week"] <= "2011-05-30"]
val   = model_df[(model_df["week"] >= "2011-06-27") & (model_df["week"] <= "2011-08-08")]
test  = model_df[model_df["week"] >= "2011-09-05"]

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, d.shape, d["week"].min().date(), "to", d["week"].max().date())

train (257466, 19) 2010-02-01 to 2011-05-30
val (29780, 19) 2011-06-27 to 2011-08-08
test (44154, 19) 2011-09-05 to 2011-11-07


The 3-week gaps are there because each row’s target covers 4 weeks. Without a gap, training labels would include sales from the validation or test period.

In [2]:
#Metrix Function
def evaluate(y_true, y_pred):
    err = y_pred - y_true
    mae = np.abs(err).mean()
    rmse = np.sqrt((err ** 2).mean())
    wape = np.abs(err).sum() / y_true.sum()
    return {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "WAPE": round(wape, 3)}

WAPE is total absolute error divided by total actual units. It is easy to explain to a business user.

In [3]:
#Baselines 
results = {}

# Baseline 1: naive, next 4 weeks = last week x 4
results["Naive (4 x lag1)"] = evaluate(val["target"], 4 * val["units_lag1"])

# Baseline 2: moving average, next 4 weeks = mean of last 4 weeks x 4
results["Moving avg (4 x mean4)"] = evaluate(val["target"], 4 * val["units_mean4"])

pd.DataFrame(results).T

,MAE,RMSE,WAPE
Naive (4 x lag1),63.64,267.17,0.763
Moving avg (4 x mean4),46.52,166.12,0.558


The 4-week moving average is the stronger baseline, since averaging smooths out single-week spikes.”